# Chapter 9: Functions — Basics to Advanced

### Why This Matters

A **function** packages a piece of logic under a name so you can reuse it without copy-pasting. This single idea — write once, call many times — is the difference between a script that's 500 lines of repeated near-identical blocks and one that's 50 lines of clean, reusable logic. It's also the foundation everything else in this course builds on: every scikit-learn model, every Pandas method, every library function you'll call from Volume 02 onward is, underneath, exactly this same concept — someone else already wrote the function; you just call it.

### 9.1 Defining and Calling Functions

In [ ]:
def calculate_profit(revenue, cost):
    """Return profit and margin."""      # a docstring — documents what the function does
    profit = revenue - cost
    margin = (profit / revenue) * 100 if revenue else 0     # guards against division by zero
    return profit, margin

p, m = calculate_profit(1500000, 1100000)      # returning multiple values is really returning ONE tuple
print(f"Profit: ₹{p:,.0f} | Margin: {m:.1f}%")

Notice `return profit, margin` — Python doesn't have true "multiple return values"; it packages them into a tuple, and the calling line `p, m = calculate_profit(...)` unpacks that tuple, exactly as you learned in Chapter 5. Recognizing this connection is a small thing that makes the language feel far more coherent rather than like a pile of unrelated features.

The **docstring** (the triple-quoted string right after `def`) isn't a comment — it's metadata that tools like `help(calculate_profit)`, Jupyter's `?` inspector, and IDE tooltips all read to show documentation. Get in the habit of writing one for any function whose purpose isn't blindingly obvious from its name and arguments.

### 9.2 Default Arguments, `*args`, and `**kwargs`

In [ ]:
def greet(name, title="Analyst"):        # "Analyst" is the DEFAULT — used if caller doesn't supply title
    return f"Hello {title} {name}"

print(greet("Priya"))                     # uses the default title
print(greet("Rahul", title="Manager"))    # overrides it

def total_sales(*amounts):                # *args collects any number of positional arguments into a tuple
    return sum(amounts)

print(total_sales(100, 200, 300))         # amounts becomes (100, 200, 300) inside the function

def create_report(**kwargs):              # **kwargs collects any number of keyword arguments into a dict
    for k, v in kwargs.items():
        print(f"{k}: {v}")

create_report(region="North", sales=1250000, target=1200000)

This is one of the more intimidating-looking bits of Python syntax for beginners, so it's worth being explicit: `*args` and `**kwargs` are just names by convention (you could call them `*things` and `**stuff` and it would work identically) — the `*` and `**` are what matter. `*` gathers extra **positional** arguments into a tuple; `**` gathers extra **keyword** arguments into a dictionary. You'll see this exact pattern in nearly every scikit-learn and Pandas function signature you encounter from Volume 02 onward (e.g., `pd.read_csv(path, **kwargs)` passing through dozens of optional settings) — recognizing it here means it won't feel foreign later.

⚠️ **Mutable default arguments are a classic, genuinely dangerous trap** that the source notebooks don't cover but you will eventually run into:

In [ ]:
# DON'T do this:
def add_item(item, cart=[]):      # the empty list is created ONCE, when the function is defined
    cart.append(item)
    return cart

print(add_item("Shoes"))     # ['Shoes']
print(add_item("Bag"))       # ['Shoes', 'Bag']  <- surprise! The previous call's list was reused

# DO this instead:
def add_item(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart

Default argument values are evaluated **once**, at function definition time, not on every call. If that default is a mutable object (a list or dict), every call that doesn't supply its own value shares and mutates the *same* object. The fix — using `None` as the default and creating the real empty container inside the function body — is a standard, memorizable idiom.

### 9.3 Lambda, map, and filter

In [ ]:
sales = [85, 120, 95, 140]
squared = list(map(lambda x: x**2, sales))       # apply a function to every element
high = list(filter(lambda x: x >= 100, sales))   # keep only elements where the function returns True
print(squared)
print(high)

A **lambda** is a small, anonymous (unnamed) function, restricted to a single expression. `lambda x: x**2` is exactly equivalent to:

In [ ]:
def square(x):
    return x**2

...just without giving it a name, which is useful when you need a throwaway function purely as an argument to another function (like `map` or `filter` here, or `sorted(data, key=lambda row: row["sales"])` later). 💡 **In modern, idiomatic Python, list comprehensions are usually preferred over `map`/`filter` for readability** — `[x**2 for x in sales]` and `[x for x in sales if x >= 100]` do the identical job as the two lines above and most Python style guides recommend them. `map`/`filter` still matter because you'll see them in older codebases and because `lambda` itself (independent of `map`/`filter`) is used constantly as the `key=` argument to `sorted()`, and inside Pandas' `.apply()`.

### Cheat Sheet — Chapter 9

| Concept | Syntax |
|---|---|
| Define | `def name(params):` |
| Default value | `def f(x, y=10):` |
| Variable positional args | `def f(*args):` → tuple |
| Variable keyword args | `def f(**kwargs):` → dict |
| Multiple returns | `return a, b` (packs into a tuple) |
| Anonymous function | `lambda x: expr` |
| Apply to all | `map(func, iterable)` |
| Keep matching | `filter(func, iterable)` |

### 🎯 Business Challenge — Chapter 9

In [ ]:
def commission(sales, rate=0.05):
    return sales * rate

def total_commission(*sales_figures, rate=0.05):
    return sum(commission(s, rate) for s in sales_figures)

print(commission(200000))                              # 10000.0
print(total_commission(200000, 150000, 300000))        # 32500.0
print(total_commission(200000, 150000, rate=0.08))     # 28000.0

---